# DESC ELAsTiCC2 — 03 : Fit Bazin multi-bandes + extraction de features pour classification

Ce notebook reprend la logique d'ajustement individuel de `02_elaticc2_fit_bazin_lightcurves.ipynb`
et l'étend pour :

1. boucler sur **plusieurs classes d'objets SNANA** (`OBJ_CLASS_LIST`),
2. fitter la fonction de Bazin **bande par bande** pour chaque objet sélectionné,
3. aplatir chaque résultat de fit en **une ligne de tableau** (features + redshift + identifiants + flag qualité),
4. sauvegarder **un fichier parquet par type de supernova** dans `features/`.

Ces fichiers parquet seront relus dans un notebook ultérieur pour construire les échantillons d'entraînement et de test du classifieur.

- author : Sylvie Dagoret-Campagne
- creation date : 2026-06-20
- derived from : `02_elaticc2_fit_bazin_lightcurves.ipynb`

## 0 · Imports

In [1]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.special import gamma as gamma_func

# ── local library (src layout) ───────────────────────────────────────────────
# notebook is in notebooks/03_fitbazinfunc/ → repo root is parent.parent
repo_root = pathlib.Path(os.getcwd()).parent.parent
srcdir = repo_root / "src"
sys.path.insert(0, str(srcdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

[2026-06-21 14:11:39 - INFO] - Imports done.


In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "x-large",
    "figure.figsize": (10, 6),
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "x-large",
    "ytick.labelsize": "x-large",
}
plt.rcParams.update(params)

In [2]:
# Imports des sous-modules
from transcientslightcurves.bazinfunction.fitting import fit_single_event, fit_bazin_band
from transcientslightcurves.bazinfunction.models import bazin_function
from transcientslightcurves.bazinfunction.utils import filter_valid_events, filter_valid_points
from transcientslightcurves.constants import BANDS, ZERO_POINT

## 1 · Paramètres

- `OBJ_CLASS_LIST` : liste des classes SNANA à traiter (une boucle complète + une sauvegarde parquet par classe).
- `N_CURVES` : nombre maximal d'objets fittés par classe (sélection aléatoire parmi les objets valides).
- `Z_MIN`, `Z_MAX` : intervalle de redshift (ZCMB) pour la présélection.
- `FILE_NUM` : numéro du fichier PHOT chargé (1–40 ; `None` = tous les fichiers, plus lent).
- `MIN_DETECTIONS` : nombre minimal de détections (PHOTFLAG détecté) exigé par objet en présélection.
- `MIN_BANDS`, `MIN_POINTS`, `MIN_TOTAL_POINTS` : critères de `filter_valid_events`.
- `DETECTED_ONLY` : si `True`, seuls les points détectés sont utilisés pour le fit.
- `OUTPUT_DIR` : dossier où sont écrits les fichiers parquet (un par classe).

In [3]:
# ── Paramètres principaux ────────────────────────────────────────────────────

OBJ_CLASS_LIST = [
    'SNIa-SALT3',
    'SNIb-Templates',
    'SNIc-Templates',
    'SNII-Templates',
]

N_CURVES        = 100000       # Nombre max d'objets fittés par classe
Z_MIN           = 0.01        # Borne inférieure du redshift (ZCMB)
Z_MAX           = 3.5         # Borne supérieure du redshift (ZCMB)
FILE_NUM        = 1           # Fichier PHOT à charger (1–40 ; None = tous)
MIN_DETECTIONS  = 5           # Nombre minimum de détections par objet (présélection)
DETECTED_ONLY   = True        # True : points détectés uniquement pour le fit
RANDOM_SEED     = 42          # None pour aléatoire

MIN_BANDS         = 3         # Nombre minimal de bandes avec assez de points
MIN_POINTS         = 3        # Nombre minimal de points par bande
MIN_TOTAL_POINTS   = MIN_DETECTIONS   # Nombre minimal total de points (toutes bandes confondues)

# Chemin vers les données ELAsTiCC2
DATA_DIR    = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX  = "ELASTICC2_TRAIN_02_"

# Dossier de sortie pour les fichiers de features (un parquet par classe)
OUTPUT_DIR = pathlib.Path(os.getcwd()) / "features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Coupures qualité (cf. Dai, Kuhlmann, Wang & Kovacs 2017, arXiv:1701.05689) ──
# Utilisées pour calculer le flag `is_good_fit`, PAS pour filtrer les données sauvegardées.
QC_CHI2_RED_MAX = 10.0
QC_B_MIN, QC_B_MAX = -20.0, 20.0
QC_T_FALL_MAX = 150.0
QC_T_RISE_MIN = 1.0
QC_T_RISE_TOL = 0.01     # tolérance autour de t_rise == 1 (valeur de bord considérée suspecte)
QC_A_MAX = 5000.0
QC_A_MAX_UY = 1000.0     # seuil plus strict pour les bandes u et Y

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"Classes à traiter : {OBJ_CLASS_LIST}")
print(f"Sortie : {OUTPUT_DIR}")

Classes à traiter : ['SNIa-SALT3', 'SNIb-Templates', 'SNIc-Templates', 'SNII-Templates']
Sortie : /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features


## 2 · Rappel : modèle de Bazin et méthodologie

$$
f(t)=A \exp[-(t-t_0)/t_{fall}] \,/\, [1+\exp[-(t-t_0)/t_{rise}]]+B
$$

- $t_{rise}$ : temps caractéristique de montée.
- $t_{fall}$ : temps caractéristique de décroissance.
- $A$ : amplitude (liée à la luminosité au pic).
- $B$ : offset (flux résiduel).
- $t_0$ : temps de référence (proche du pic).

Pour chaque objet, `fit_single_event` ajuste indépendamment chaque bande (`u,g,r,i,z,Y`), calcule
`t_max`, `f_max`, `m_p` par bande, les couleurs au pic `c_ij = m_p,i - m_p,j` entre bandes adjacentes,
et un `t_max` / `F_peak` global (médiane des bandes réussies).

Référence : *Photometric classification and redshift estimation of LSST Supernovae*, Dai, Kuhlmann, Wang & Kovacs, https://arxiv.org/pdf/1701.05689

## 3 · Fonctions utilitaires : sélection, fit en boucle, aplatissement (flatten) et flag qualité

In [4]:
def select_valid_snids(esr, obj_class, z_min, z_max, min_detections,
                        min_bands, min_points, min_total_points,
                        n_curves, file_num, rng):
    """
    Charge HEAD/truth/lightcurves pour une classe SNANA, applique les filtres
    de redshift / nombre de détections / nombre de bandes valides, puis tire
    aléatoirement jusqu'à `n_curves` SNID parmi les objets valides.

    Retourne (chosen_snids, all_ltcvs, truth, head).
    """
    _logger.info(f"[{obj_class}] Loading HEAD...")
    head = esr.get_head(obj_class, return_format='pandas')
    _logger.info(f"[{obj_class}] Loading truth...")
    truth = esr.get_object_truth(obj_class, return_format='pandas')
    _logger.info(f"[{obj_class}] Loading light curves (file_num={file_num})...")
    all_ltcvs = esr.get_all_ltcvs(obj_class, file_num=file_num, return_format='pandas')
    _logger.info(f"[{obj_class}] {all_ltcvs['SNID'].nunique()} objects loaded.")

    detcounts = (
        all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
        .groupby('SNID').agg('count')['MJD']
        .reset_index()
        .rename({'MJD': 'ndetect'}, axis=1)
    )

    truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
    subset = truth_counts[
        (truth_counts['ZCMB'] >= z_min) &
        (truth_counts['ZCMB'] <  z_max) &
        (truth_counts['ndetect'] >= min_detections)
    ].copy()
    _logger.info(f"[{obj_class}] {len(subset)} objects pass z/ndetect selection.")

    valid_subset = filter_valid_events(
        subset, all_ltcvs,
        min_bands=min_bands, min_points=min_points, min_total_points=min_total_points
    )
    _logger.info(f"[{obj_class}] {len(valid_subset)} objects pass band/point quality filter.")

    if len(valid_subset) == 0:
        return np.array([]), all_ltcvs, truth, head

    n_avail = min(n_curves, len(valid_subset))
    chosen_idx = rng.choice(len(valid_subset), size=n_avail, replace=False)
    chosen_snids = valid_subset['SNID'].values[chosen_idx]

    return chosen_snids, all_ltcvs, truth, head

In [5]:
def fit_events(chosen_snids, all_ltcvs, esr, detected_only=True, verbose=False):
    """
    Boucle sur `chosen_snids`, applique les filtres de détection/SNR, et
    appelle `fit_single_event` pour chacun. Retourne un dict {snid: result}.
    """
    fit_results = {}
    for idx, snid in enumerate(chosen_snids, start=1):
        ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]

        if detected_only:
            ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

        mask_det = (
            (ltcv['FLUXCALERR'] > 0) &
            (ltcv['FLUXCAL'] > 0) &
            (ltcv['FLUXCAL'] / ltcv['FLUXCALERR'] > 3)
        )
        ltcv = ltcv[mask_det]

        if len(ltcv) == 0:
            fit_results[snid] = {'success': False, 'error': 'Aucun point valide après filtre.'}
            continue

        try:
            result = fit_single_event(ltcv)
        except Exception as e:
            fit_results[snid] = {'success': False, 'error': f'Erreur: {str(e)}'}
            continue

        fit_results[snid] = result

        if verbose:
            status = '✓' if result.get('success') else '✗'
            chi2r = result.get('chi2_red', float('nan'))
            print(f"  [{idx}/{len(chosen_snids)}] SNID {snid:10d}  {status}  χ²/dof = {chi2r:.2f}")

    return fit_results

### Flag qualité

Critères repris de l'article (Section 3.3), appliqués **par bande puis combinés** :

- `t_rise > 1` et pas trop proche de la borne (`|t_rise - 1| > tol`),
- `-20 < B < 20`,
- `χ²/ndof < 10` (par bande, pour les bandes ajustées),
- `t_fall < 150`,
- `t_rise < t_fall`,
- `A < 5000` (`A < 1000` pour les bandes `u` et `Y`),

Le flag global `is_good_fit` est `True` si le fit global a réussi **et** si toutes les bandes
effectivement ajustées (par Bazin complet, pas par constante) respectent ces critères.
Cette colonne n'est **pas** utilisée pour filtrer ici — elle est sauvegardée pour permettre
un filtrage flexible dans le notebook d'entraînement.

In [6]:
def check_band_quality(band_result):
    """
    Applique les coupures qualité de Dai et al. 2017 à un seul résultat de bande.
    Retourne True si la bande est un 'bon' fit Bazin complet (pas un fit constant,
    pas un échec), False sinon. Les bandes non ajustées (échec) ne comptent pas
    contre le flag global (elles sont simplement absentes des features).
    """
    if not band_result.get('success', False):
        return None  # bande non disponible : ne pénalise pas, ne valide pas non plus

    params = band_result['params']
    ndof = band_result.get('ndof', 0)

    # Fit constant (1 point, ndof == 0, t_rise == t_fall == 0) : on l'ignore du QC strict
    if ndof == 0 and params['t_rise'] == 0 and params['t_fall'] == 0:
        return None

    chi2 = band_result.get('chi2', np.nan)
    chi2_red = chi2 / ndof if ndof > 0 else np.nan

    band_name = band_result.get('_band', None)
    a_max = QC_A_MAX_UY if band_name in ('u', 'Y') else QC_A_MAX

    ok = (
        (params['t_rise'] > QC_T_RISE_MIN) and
        (abs(params['t_rise'] - QC_T_RISE_MIN) > QC_T_RISE_TOL) and
        (QC_B_MIN < params['B'] < QC_B_MAX) and
        (not np.isnan(chi2_red)) and (chi2_red < QC_CHI2_RED_MAX) and
        (params['t_fall'] < QC_T_FALL_MAX) and
        (params['t_rise'] < params['t_fall']) and
        (params['A'] < a_max)
    )
    return bool(ok)


def compute_is_good_fit(result):
    """
    Combine le QC par bande en un flag global. True seulement si le fit global
    a réussi et qu'au moins une bande a été validée par check_band_quality,
    et qu'aucune bande complètement ajustée n'a échoué les coupures.
    """
    if not result.get('success', False):
        return False

    band_flags = []
    for band in BANDS:
        band_result = dict(result['band_params'][band])
        band_result['_band'] = band
        flag = check_band_quality(band_result)
        if flag is not None:
            band_flags.append(flag)

    if len(band_flags) == 0:
        return False

    return bool(np.all(band_flags))

### Aplatissement (flatten) d'un résultat de fit en une ligne de tableau

In [7]:
def flatten_fit_result(snid, obj_class, result, truth_row, head_row=None):
    """
    Convertit un résultat de `fit_single_event` (+ métadonnées truth/head) en
    un dict plat, prêt à devenir une ligne de DataFrame.

    Colonnes produites :
      - identifiants : SNID, obj_class
      - vérité : redshift (ZCMB), type SNANA si disponible (GENTYPE/SNTYPE), PEAKMJD si disponible
      - qualité globale : fit_success, chi2_total, ndof_total, chi2_red, is_good_fit
      - globaux : t_max_global, F_peak_global
      - par bande (pour chaque bande de BANDS) : {band}_A, {band}_t0, {band}_t_fall,
        {band}_t_rise, {band}_B, {band}_t_max, {band}_f_max, {band}_m_p, {band}_chi2,
        {band}_ndof, {band}_chi2_red, {band}_success, {band}_n_points
      - couleurs : c_{band1}{band2} pour les paires de bandes adjacentes
    """
    row = {
        'SNID': int(snid),
        'obj_class': obj_class,
    }

    # ── Vérité (redshift + type + autres colonnes utiles si présentes) ──────
    row['redshift'] = float(truth_row['ZCMB']) if 'ZCMB' in truth_row else np.nan
    for col in ('GENTYPE', 'SNTYPE', 'PEAKMJD', 'MWEBV', 'GENSOURCE', 'NON1A_INDEX'):
        if col in truth_row:
            row[f'truth_{col}'] = truth_row[col]

    if head_row is not None:
        for col in ('RA', 'DEC', 'MWEBV', 'REDSHIFT_HELIO', 'REDSHIFT_FINAL'):
            if col in head_row and f'truth_{col}' not in row:
                row[col.lower()] = head_row[col]

    # ── Statut global du fit ─────────────────────────────────────────────────
    row['fit_success'] = bool(result.get('success', False))

    if not row['fit_success']:
        row['is_good_fit'] = False
        row['chi2_total'] = np.nan
        row['ndof_total'] = np.nan
        row['chi2_red'] = np.nan
        row['t_max_global'] = np.nan
        row['F_peak_global'] = np.nan
        for band in BANDS:
            for suffix in ('A', 't0', 't_fall', 't_rise', 'B', 't_max', 'f_max',
                           'm_p', 'chi2', 'ndof', 'chi2_red'):
                row[f'{band}_{suffix}'] = np.nan
            row[f'{band}_success'] = False
        for i in range(len(BANDS) - 1):
            row[f'c_{BANDS[i]}{BANDS[i+1]}'] = np.nan
        return row

    row['is_good_fit'] = compute_is_good_fit(result)
    row['chi2_total'] = result.get('chi2_total', np.nan)
    row['ndof_total'] = result.get('ndof_total', np.nan)
    row['chi2_red'] = result.get('chi2_red', np.nan)
    row['t_max_global'] = result['global_params'].get('t_max', np.nan)
    row['F_peak_global'] = result['global_params'].get('F_peak', np.nan)

    for band in BANDS:
        band_result = result['band_params'][band]
        success = band_result.get('success', False)
        row[f'{band}_success'] = bool(success)
        if success:
            params = band_result['params']
            ndof = band_result.get('ndof', np.nan)
            chi2 = band_result.get('chi2', np.nan)
            row[f'{band}_A'] = params['A']
            row[f'{band}_t0'] = params['t0']
            row[f'{band}_t_fall'] = params['t_fall']
            row[f'{band}_t_rise'] = params['t_rise']
            row[f'{band}_B'] = params['B']
            row[f'{band}_t_max'] = params['t_max']
            row[f'{band}_f_max'] = params['f_max']
            row[f'{band}_m_p'] = params['m_p']
            row[f'{band}_chi2'] = chi2
            row[f'{band}_ndof'] = ndof
            row[f'{band}_chi2_red'] = (chi2 / ndof) if (ndof and ndof > 0) else np.nan
        else:
            for suffix in ('A', 't0', 't_fall', 't_rise', 'B', 't_max', 'f_max',
                           'm_p', 'chi2', 'ndof', 'chi2_red'):
                row[f'{band}_{suffix}'] = np.nan

    colors = result.get('colors', {})
    for i in range(len(BANDS) - 1):
        key = f'c_{BANDS[i]}{BANDS[i+1]}'
        row[key] = colors.get(key, np.nan)

    return row

## 4 · Boucle principale : pour chaque classe, sélectionner, fitter, aplatir, sauvegarder

In [8]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)
print(f"Classes disponibles dans {DATA_DIR} :")
print(esr.obj_class_names)

Classes disponibles dans /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2 :
['SNII-Templates', 'SNIa-SALT3', 'SNIb-Templates', 'SNIc-Templates']


In [9]:
summary = {}          # obj_class -> dict de stats résumées
output_files = {}     # obj_class -> chemin du parquet écrit

for obj_class in OBJ_CLASS_LIST:
    print(f"\n{'='*70}")
    print(f"  Classe : {obj_class}")
    print(f"{'='*70}")

    chosen_snids, all_ltcvs, truth, head = select_valid_snids(
        esr, obj_class,
        z_min=Z_MIN, z_max=Z_MAX, min_detections=MIN_DETECTIONS,
        min_bands=MIN_BANDS, min_points=MIN_POINTS, min_total_points=MIN_TOTAL_POINTS,
        n_curves=N_CURVES, file_num=FILE_NUM, rng=rng
    )

    if len(chosen_snids) == 0:
        print(f"  ⚠ Aucun objet valide pour {obj_class}, on passe à la classe suivante.")
        summary[obj_class] = {'n_selected': 0, 'n_fitted_ok': 0, 'n_good_fit': 0}
        continue

    print(f"  {len(chosen_snids)} SNID sélectionnés pour le fit.")

    fit_results = fit_events(chosen_snids, all_ltcvs, esr, detected_only=DETECTED_ONLY, verbose=False)

    truth_indexed = truth.set_index('SNID')
    head_indexed = head.set_index('SNID') if head is not None else None

    rows = []
    for snid in chosen_snids:
        result = fit_results.get(snid, {'success': False, 'error': 'missing'})
        truth_row = truth_indexed.loc[snid] if snid in truth_indexed.index else pd.Series(dtype=object)
        head_row = head_indexed.loc[snid] if (head_indexed is not None and snid in head_indexed.index) else None
        row = flatten_fit_result(snid, obj_class, result, truth_row, head_row)
        rows.append(row)

    df_features = pd.DataFrame(rows)

    n_fitted_ok = int(df_features['fit_success'].sum())
    n_good_fit = int(df_features['is_good_fit'].sum())
    print(f"  Fits réussis : {n_fitted_ok}/{len(df_features)}  —  is_good_fit=True : {n_good_fit}/{len(df_features)}")

    # ── métadonnées de provenance ────────────────────────────────────────────
    df_features.attrs['obj_class'] = obj_class
    df_features.attrs['data_dir'] = DATA_DIR
    df_features.attrs['file_num'] = FILE_NUM
    df_features.attrs['z_min'] = Z_MIN
    df_features.attrs['z_max'] = Z_MAX
    df_features.attrs['created_utc'] = datetime.now(timezone.utc).isoformat()

    out_path = OUTPUT_DIR / f"bazin_features_{obj_class}.parquet"
    df_features.to_parquet(out_path, index=False)
    print(f"  → écrit : {out_path}")

    output_files[obj_class] = out_path
    summary[obj_class] = {
        'n_selected': len(chosen_snids),
        'n_fitted_ok': n_fitted_ok,
        'n_good_fit': n_good_fit,
    }

print("\nTerminé.")

[2026-06-21 14:11:39 - INFO] - [SNIa-SALT3] Loading HEAD...
[2026-06-21 14:11:39 - INFO] - Reading HEAD files from /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNIa-SALT3



  Classe : SNIa-SALT3


[2026-06-21 14:11:41 - INFO] - [SNIa-SALT3] Loading truth...
[2026-06-21 14:11:41 - INFO] - Reading /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNIa-SALT3/ELASTICC2_TRAIN_02_SNIa-SALT3.DUMP
[2026-06-21 14:11:42 - INFO] - [SNIa-SALT3] Loading light curves (file_num=1)...
[2026-06-21 14:11:42 - INFO] - Reading /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNIa-SALT3/ELASTICC2_TRAIN_02_NONIaMODEL0-0001_PHOT.FITS.gz...
[2026-06-21 14:11:43 - INFO] - ...assigning SNID
[2026-06-21 14:11:43 - DEBUG] - Concatenating 1 dataframes
[2026-06-21 14:11:43 - DEBUG] - Sorting
[2026-06-21 14:11:43 - DEBUG] - Returning
[2026-06-21 14:11:43 - INFO] - [SNIa-SALT3] 4247 objects loaded.
[2026-06-21 14:11:43 - INFO] - [SNIa-SALT3] 1892 objects pass z/ndetect selection.
[2026-06-21 14:11:44 - INFO] - [SNIa-SALT3] 1296 objects pass band/point quality filter.


  1296 SNID sélectionnés pour le fit.


[2026-06-21 14:12:43 - INFO] - [SNIb-Templates] Loading HEAD...
[2026-06-21 14:12:43 - INFO] - Reading HEAD files from /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNIb-Templates


...

  104 SNID sélectionnés pour le fit.


[2026-06-21 14:13:04 - INFO] - [SNII-Templates] Loading HEAD...
[2026-06-21 14:13:04 - INFO] - Reading HEAD files from /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNII-Templates


  Fits réussis : 45/104  —  is_good_fit=True : 25/104
  → écrit : /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features/bazin_features_SNIc-Templates.parquet

  Classe : SNII-Templates


[2026-06-21 14:13:06 - INFO] - [SNII-Templates] Loading truth...
[2026-06-21 14:13:06 - INFO] - Reading /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNII-Templates/ELASTICC2_TRAIN_02_SNII-Templates.DUMP
[2026-06-21 14:13:06 - INFO] - [SNII-Templates] Loading light curves (file_num=1)...
[2026-06-21 14:13:06 - INFO] - Reading /Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/ELASTICC2_TRAIN_02_SNII-Templates/ELASTICC2_TRAIN_02_NONIaMODEL0-0001_PHOT.FITS.gz...
[2026-06-21 14:13:07 - INFO] - ...assigning SNID
[2026-06-21 14:13:07 - DEBUG] - Concatenating 1 dataframes
[2026-06-21 14:13:07 - DEBUG] - Sorting
[2026-06-21 14:13:07 - DEBUG] - Returning
[2026-06-21 14:13:07 - INFO] - [SNII-Templates] 1920 objects loaded.
[2026-06-21 14:13:07 - INFO] - [SNII-Templates] 688 objects pass z/ndetect selection.
[2026-06-21 14:13:07 - INFO] - [SNII-Templates] 491 objects pass band/point quality filter.


  491 SNID sélectionnés pour le fit.
  Fits réussis : 264/491  —  is_good_fit=True : 125/491
  → écrit : /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features/bazin_features_SNII-Templates.parquet

Terminé.


## 5 · Résumé

In [10]:
summary_df = pd.DataFrame(summary).T
summary_df.index.name = 'obj_class'
summary_df

,n_selected,n_fitted_ok,n_good_fit
obj_class,,,
SNIa-SALT3,1296,696,386
SNIb-Templates,209,100,54
SNIc-Templates,104,45,25
SNII-Templates,491,264,125


In [11]:
print("Fichiers parquet écrits :")
for obj_class, path in output_files.items():
    print(f"  {obj_class:20s} → {path}")

Fichiers parquet écrits :
  SNIa-SALT3           → /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features/bazin_features_SNIa-SALT3.parquet
  SNIb-Templates       → /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features/bazin_features_SNIb-Templates.parquet
  SNIc-Templates       → /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features/bazin_features_SNIc-Templates.parquet
  SNII-Templates       → /Users/dagoret/Desktop/transcientslightcurves/notebooks/03_fitbazinfunc/features/bazin_features_SNII-Templates.parquet


## 6 · Vérification rapide : relecture d'un fichier

Aperçu du contenu d'un des fichiers parquet sauvegardés (utile pour vérifier les colonnes
avant d'écrire le notebook de construction des échantillons train/test).

In [12]:
if output_files:
    first_class = next(iter(output_files))
    df_check = pd.read_parquet(output_files[first_class])
    print(f"Classe : {first_class}  —  shape={df_check.shape}")
    display(df_check.head())
    print("\nColonnes :")
    print(list(df_check.columns))

Classe : SNIa-SALT3  —  shape=(1296, 95)


,SNID,obj_class,redshift,truth_GENTYPE,truth_PEAKMJD,truth_MWEBV,truth_NON1A_INDEX,ra,dec,redshift_helio,...,Y_m_p,Y_chi2,Y_ndof,Y_chi2_red,Y_success,c_ug,c_gr,c_ri,c_iz,c_zY
0,41010627,SNIa-SALT3,0.623990,10.0,60877.589,0.031959,0.0,8.066771,-6.602528,0.625868,...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
1,62241130,SNIa-SALT3,0.605364,10.0,60563.379,0.024797,0.0,344.796530,-68.331154,0.901390,...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
2,36639284,SNIa-SALT3,0.297030,10.0,61237.617,0.326777,0.0,268.597875,-37.727200,0.296667,...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
3,72833645,SNIa-SALT3,0.568437,10.0,60634.273,0.018060,0.0,38.938811,0.290894,0.672814,...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
4,17214509,SNIa-SALT3,0.304530,10.0,61019.392,0.015721,0.0,8.974761,-39.213664,0.303919,...,NaN,0.0,0.0,NaN,True,NaN,NaN,NaN,-0.187049,NaN



Colonnes :
['SNID', 'obj_class', 'redshift', 'truth_GENTYPE', 'truth_PEAKMJD', 'truth_MWEBV', 'truth_NON1A_INDEX', 'ra', 'dec', 'redshift_helio', 'redshift_final', 'fit_success', 'is_good_fit', 'chi2_total', 'ndof_total', 'chi2_red', 't_max_global', 'F_peak_global', 'u_A', 'u_t0', 'u_t_fall', 'u_t_rise', 'u_B', 'u_t_max', 'u_f_max', 'u_m_p', 'u_chi2', 'u_ndof', 'u_chi2_red', 'u_success', 'g_A', 'g_t0', 'g_t_fall', 'g_t_rise', 'g_B', 'g_t_max', 'g_f_max', 'g_m_p', 'g_chi2', 'g_ndof', 'g_chi2_red', 'g_success', 'r_A', 'r_t0', 'r_t_fall', 'r_t_rise', 'r_B', 'r_t_max', 'r_f_max', 'r_m_p', 'r_chi2', 'r_ndof', 'r_chi2_red', 'r_success', 'i_A', 'i_t0', 'i_t_fall', 'i_t_rise', 'i_B', 'i_t_max', 'i_f_max', 'i_m_p', 'i_chi2', 'i_ndof', 'i_chi2_red', 'i_success', 'z_A', 'z_t0', 'z_t_fall', 'z_t_rise', 'z_B', 'z_t_max', 'z_f_max', 'z_m_p', 'z_chi2', 'z_ndof', 'z_chi2_red', 'z_success', 'Y_A', 'Y_t0', 'Y_t_fall', 'Y_t_rise', 'Y_B', 'Y_t_max', 'Y_f_max', 'Y_m_p', 'Y_chi2', 'Y_ndof', 'Y_chi2_red', '